## **Seg Former-B0 Modern  model**

In [ ]:
import os
import time
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras
import keras_hub

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("KerasHub:", keras_hub.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("Available GPUs:", gpus)

if not gpus:
    print("WARNING: No GPU detected. Enable GPU from Kaggle Settings.")

TensorFlow: 2.21.0
Keras: 3.13.2
KerasHub: 0.26.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [ ]:
# DATASET VERIFICATION

sample_images, sample_masks = next(iter(train_ds))

print("Images shape:", sample_images.shape)
print("Images dtype:", sample_images.dtype)

print("Masks shape:", sample_masks.shape)
print("Masks dtype:", sample_masks.dtype)

print(
    "Image value range:",
    tf.reduce_min(sample_images).numpy(),
    tf.reduce_max(sample_images).numpy()
)

print(
    "Mask classes:",
    np.unique(sample_masks.numpy())
)

assert sample_images.shape[1:] == (512, 512, 3)

assert sample_masks.shape[1:] == (512, 512)

assert sample_masks.dtype in [
    tf.int32,
    tf.int64,
    tf.uint8
]

assert len(CLASS_NAMES) == NUM_CLASSES

print("\nDataset verification completed successfully.")


Images shape: (8, 512, 512, 3)
Images dtype: <dtype: 'float32'>
Masks shape: (8, 512, 512)
Masks dtype: <dtype: 'int32'>
Image value range: -2.117904 2.4285715
Mask classes: [0 2 4]

Dataset verification completed successfully.


In [ ]:
keras.backend.clear_session()

keras.utils.set_random_seed(SEED)

# 1. Load pretrained MiT-B0 with the correct input size
mit_encoder = keras_hub.models.MiTBackbone.from_preset(
    "mit_b0_ade20k_512",
    load_weights=True,
    image_shape=(512, 512, 3)
)

print("Encoder input shape:", mit_encoder.input_shape)


# 2. Build SegFormer backbone
segformer_backbone = keras_hub.models.SegFormerBackbone(
    image_encoder=mit_encoder,
    projection_filters=256
)


# 3. Build segmentation model for 5 classes
segformer_model = keras_hub.models.SegFormerImageSegmenter(
    backbone=segformer_backbone,
    num_classes=NUM_CLASSES
)


# 4. Verify model input and output
print("Model input shape:", segformer_model.input_shape)

sample_logits = segformer_model(
    sample_images[:1],
    training=False
)

print("Model output shape:", sample_logits.shape)


# 5. Verify expected dimensions
assert segformer_model.input_shape[1:] == (512, 512, 3)

assert sample_logits.shape[1:] == (512, 512, NUM_CLASSES)

print("SegFormer-B0 model built successfully")

Encoder input shape: (None, 512, 512, 3)
Model input shape: (None, 512, 512, 3)
Model output shape: (1, 512, 512, 5)
SegFormer-B0 model built successfully


In [ ]:
LEARNING_RATE = 6e-5

segformer_model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=1e-4,
        clipnorm=1.0
    ),

    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),

    metrics=[
        keras.metrics.SparseCategoricalAccuracy(
            name="pixel_accuracy"
        )
    ]
)

print("SegFormer-B0 compiled successfully.")

SegFormer-B0 compiled successfully.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Training
EPOCHS = 10

CHECKPOINT_PATH = (
    "/kaggle/working/segformer_b0_best.weights.h5"
)

callbacks = [

    keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        mode="min",
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

training_start = time.perf_counter()

history = segformer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

training_time = time.perf_counter() - training_start

# Explicitly load the best saved weights
segformer_model.load_weights(CHECKPOINT_PATH)

print("\nTraining completed.")
print("Training time in minutes:", round(training_time / 60, 2))

Epoch 1/10
933/933 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - loss: 0.6171 - pixel_accuracy: 0.8132

E0000 00:00:1795044427.801175     143 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795044427.971323     143 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795044429.047983     143 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795044429.223007     143 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.



Epoch 1: val_loss improved from None to 0.19481, saving model to /kaggle/working/segformer_b0_best.weights.h5

Epoch 1: finished saving model to /kaggle/working/segformer_b0_best.weights.h5
933/933 ━━━━━━━━━━━━━━━━━━━━ 399s 380ms/step - loss: 0.3857 - pixel_accuracy: 0.8819 - val_loss: 0.1948 - val_pixel_accuracy: 0.9348 - learning_rate: 6.0000e-05
Epoch 2/10
933/933 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step - loss: 0.2112 - pixel_accuracy: 0.9307
Epoch 2: val_loss improved from 0.19481 to 0.17144, saving model to /kaggle/working/segformer_b0_best.weights.h5

Epoch 2: finished saving model to /kaggle/working/segformer_b0_best.weights.h5
933/933 ━━━━━━━━━━━━━━━━━━━━ 356s 382ms/step - loss: 0.2103 - pixel_accuracy: 0.9283 - val_loss: 0.1714 - val_pixel_accuracy: 0.9361 - learning_rate: 6.0000e-05
Epoch 3/10
933/933 ━━━━━━━━━━━━━━━━━━━━ 0s 358ms/step - loss: 0.1835 - pixel_accuracy: 0.9362
Epoch 3: val_loss improved from 0.17144 to 0.15513, saving model to /kaggle/working/segformer_b0_best.weig